In [ ]:
try:
    import claveles  # noqa: F401
except ImportError:
    %pip install claveles

**Note for Google Colab Users:**
During installation, Colab may display a dependency conflict error regarding `protobuf` (`ERROR: pip's dependency resolver...`). This occurs due to pre-installed Google packages in the Colab environment. 

As long as you see `Successfully installed ...` at the end of the log, the required packages are properly installed. **Please ignore the error and continue running the rest of the notebook.**

$$
\newcommand{\bra}[1]{{\left\langle{#1}\right|}}
\newcommand{\ket}[1]{{\left|{#1}\right\rangle}}
\newcommand{\Bra}[1]{{\Big\langle{#1}\Big|}}
\newcommand{\Ket}[1]{{\Big|{#1}\Big\rangle}}
\newcommand{\braket}[1]{{\left\langle{#1}\right\rangle}}
\newcommand{\dag}{\dagger}
$$

# Addition Using Continuous Variables

## Introduction

This demo shows how to perform scalar addition as a basic computation of continuous-variable (CV) quantum computing. Using the calculation $2 + 3 = 5$ as an example, we walk through the procedure of encoding numerical values into CV (i.e. bosonic) quantum states and realizing addition with a CV quantum gate.

## Background

In CV computing, the two **quadrature operators** of a bosonic quantum system play a central role in information processing. These operators—respectively called the **position** and **momentum** operators and denoted as $\hat{x}$ and $\hat{p}$—obey the canonical commutation relation $[\hat{x}, \hat{p}] = i\hbar$, which reflects their non-commuting quantum nature. Using their eigenstates (the $x$-quadrature basis and the $p$-quadrature basis), we can encode classical scalar values into quantum states.

$$
\ket{x = x_0} \quad\text{($x$-quadrature basis)}, \qquad \ket{p = p_0} \quad\text{($p$-quadrature basis)}
$$

As a fundamental operation in CV quantum computation, we use the CV Controlled-Z (CZ) gate. This gate is defined as $\widehat{\mathrm{CZ}}(g)=\exp(ig\hat{x}_1\hat{x}_2)$ and unitarily transforms the position eigenstates as follows:

$$
\widehat{\mathrm{CZ}}(g) \bigl(\ket{p=a} \otimes \ket{x=b}\bigr) = \ket{p=a+gb} \otimes \ket{x=b}, 
\widehat{\mathrm{CZ}}(g) \bigl(\ket{x=a} \otimes \ket{p=b}\bigr) = \ket{x=a} \otimes \ket{p=ga+b}
$$

In the Heisenberg picture, the CV CZ gate transforms $\hat{x}$ and $\hat{p}$ as follows:

$$
\widehat{\mathrm{CZ}}(g) \colon (\hat{x}_1, \hat{p}_1, \hat{x}_2, \hat{p}_2) \;\mapsto\; (\hat{x}_1,\; \hat{p}_1 + g\hat{x}_2,\; \hat{x}_2,\; \hat{p}_2 + g\hat{x}_1)
$$

By setting $g=1$ and the input to $\ket{p=a} \otimes \ket{x=b}$, one obtains

$$
\widehat{\mathrm{CZ}}(1) \bigl(\ket{p=a} \otimes \ket{x=b}\bigr) = \ket{p=a+b} \otimes \ket{x=b}.
$$

We therefore expect that measuring the $p$ quadrature of the first mode yields $a + b$. However, the position and momentum eigenstates employed above can be defined only as an ideal limit and is physically unrealizable, since such eigenstates cannot be normalized in a standard Hilbert space and require infinite energy to prepare. This can also be seen from the well-known fact that all bosonic quantum states satisfy the **uncertainty principle** $\Delta x \cdot \Delta p \geq \hbar/2$, which prevents the uncertainty in either variable from ever being exactly zero [\[1,2\]](#References), while these eigenstates have a zero variance over the corresponding quadratures and thus clearly violate the principle. We therefore approximate them with physically realizable **squeezed states**:

$$
\ket{x \approx x_0}: \quad \Delta x \ll \sqrt{\frac{\hbar}{2}},\; \Delta p \gg \sqrt{\frac{\hbar}{2}} \qquad\text{($x$-squeezed state)}
$$

$$
\ket{p \approx p_0}: \quad \Delta x \gg \sqrt{\frac{\hbar}{2}},\; \Delta p \ll \sqrt{\frac{\hbar}{2}} \qquad\text{($p$-squeezed state)}
$$

The squeezed state $\ket{r}$ is formally defined using the position expansion as follows:

$$
\ket{r} = \left( \frac{1}{\pi \hbar e^{2r}} \right)^{\frac{1}{4}} \int_{-\infty}^{\infty} dx\,\exp\left( - \frac{e^{-2r}}{2\hbar} x^2 \right) \ket{x}
$$

Elementary computations confirms that this state has finite variances $\Delta x = \sqrt{\frac{\hbar}{2}}e^{-r}$ and $\Delta p = \sqrt{\frac{\hbar}{2}}e^r$; thus positive $r$ gives an $x$-squeezed state, whereas negative $r$ corresponds to a $p$-squeezed state. The squeezing parameter $r$ specifies how the variances of the two quadratures are exponentially scaled (or redistributed) relative to the vacuum state $r=0$, while keeping the state at the minimum uncertainty limit. The larger the squeezing parameter $r$, the closer the state is to the true eigenstate, and the measurement error scales as $e^{-|r|}$. 

---

## 1. Encoding Scalar Values into States

First, import the required libraries.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from claveles.circuit.state import QuantumState
from claveles.constant import hbar
from numpy import linspace

### 1.1 Creating Squeezed States

For squeezing parameter $r > 0$, the covariance matrix formalism of a squeezed vacuum state is given as follows:

- **x-squeezed** ($\Delta x = e^{-r}\sqrt{\hbar/2}$, $\Delta p = e^{r}\sqrt{\hbar/2}$): approximates an $x$-quadrature eigenstate
- **p-squeezed** ($\Delta x = e^{r}\sqrt{\hbar/2}$, $\Delta p = e^{-r}\sqrt{\hbar/2}$): approximates a $p$-quadrature eigenstate

By setting the mean to $\langle x \rangle = x_0$ for the x-squeezed state and $\langle p \rangle = p_0$ for the p-squeezed, we encode a numerical value into a squeezed coherent state.

In [ ]:
def create_x_squeezed_state(
    x0: float, r: float = 2.0, hbar: float = 1.0
) -> QuantumState:
    """x-squeezed state localized at x = x_0 (approximate x-quadrature eigenstate)"""
    cov = np.diag([np.exp(-2 * r) * hbar / 2, np.exp(2 * r) * hbar / 2])
    return QuantumState.gaussian(mean=np.array([x0, 0.0]), cov=cov)


def create_p_squeezed_state(
    p0: float, r: float = 2.0, hbar: float = 1.0
) -> QuantumState:
    """p-squeezed state localized at p = p_0 (approximate p-quadrature eigenstate)"""
    cov = np.diag([np.exp(2 * r) * hbar / 2, np.exp(-2 * r) * hbar / 2])
    return QuantumState.gaussian(mean=np.array([0.0, p0]), cov=cov)

### 1.2 Visualizing the Wigner Functions

We encode $a = 2$ into a p-squeezed state and $b = 3$ into an x-squeezed state, then visualize the Wigner function of each.

In [ ]:
a, b = 2.0, 3.0
r = 2.0

state_p = create_p_squeezed_state(p0=a, r=r, hbar=hbar)  # |p ≈ 2>  mode 1
state_x = create_x_squeezed_state(x0=b, r=r, hbar=hbar)  # |x ≈ 3>  mode 2

for state, title, fname in [
    (state_p, r"$|p \approx 2\rangle$ (mode 1)", "add_mode1_initial"),
    (state_x, r"$|x \approx 3\rangle$ (mode 2)", "add_mode2_initial"),
]:
    fig = state.draw_wigner(
        xvec=linspace(-10, 10, 401),
        pvec=linspace(-10, 10, 401),
        title=title,
    )
    plt.show()

The p-squeezed state appears in phase space as a narrow ellipse elongated along the $x$ direction, centered near $p \approx 2$. The x-squeezed state appears as a narrow ellipse elongated along the $p$ direction, centered near $x \approx 3$.

---

## 2. Addition via the CV CZ Gate

### 2.1 Implementing the CV CZ Gate

The CV CZ gate is implemented as a basic gate available from `claveles.circuit.ops.std.ControlledZ`.

In [ ]:
from claveles.circuit import CircuitRepr
from claveles.circuit.ops.std import ControlledZ, MeasureHomodyne
from claveles.client import SimulatorClient
from claveles.execute import execute

### 2.2 Visualizing the Output State

We verify the Wigner function of mode 1 after applying the CZ(1) gate to $\ket{p \approx 2} \otimes \ket{x \approx 3}$.

In [ ]:
circuit = CircuitRepr("add_vis")
circuit.Q(0) | create_p_squeezed_state(p0=a, r=r, hbar=hbar)
circuit.Q(1) | create_x_squeezed_state(x0=b, r=r, hbar=hbar)
circuit.Q(0, 1) | ControlledZ(g=1)

client = SimulatorClient(backend="cpu", state_save_policy="first_only")
client.n_shots = 1
result = execute(circuit, client)

mode1 = result.client_result.shot_results[0].state.extract_mode(0)
fig = mode1.draw_wigner(
    xvec=linspace(-10, 10, 401),
    pvec=linspace(-10, 10, 401),
    title=r"Mode 1 after CZ ($\langle p_1 \rangle \approx a + b = 5$)",
)
plt.show()

We can confirm that the output mode 1 is localized near $p \approx 5$ and forms an ellipse elongated along the $x$ direction.

### 2.3 Performing the Addition: 2 + 3 = 5

To obtain the computation result, we measure the $p$ quadrature of the first mode. By repeating the circuit over multiple trials, we can construct a histogram to verify that the distribution of measured values is centered around $a + b = 5$.

In [ ]:
def add_with_cv_cz(
    a: float, b: float, r: float = 2.0, n_shots: int = 500
) -> np.ndarray:
    circuit = CircuitRepr("add")
    circuit.Q(0) | create_p_squeezed_state(p0=a, r=r, hbar=hbar)
    circuit.Q(1) | create_x_squeezed_state(x0=b, r=r, hbar=hbar)
    circuit.Q(0, 1) | ControlledZ(g=1)
    circuit.Q(0) | MeasureHomodyne(phi=np.pi / 2)  # measure p quadrature

    client = SimulatorClient(backend="cpu", state_save_policy="first_only")
    client.n_shots = n_shots
    result = execute(circuit, client)

    return np.array([result.client_result[i].get_value(0) for i in range(n_shots)])

In [ ]:
observations = add_with_cv_cz(a, b, r=r, n_shots=1000)

print(f"Mean: {observations.mean():.3f}  (expected: {a + b:.1f})")
print(f"Std:  {observations.std():.3f}  (theory: {np.exp(-r):.3f})")

sns.set_style("whitegrid")
plt.figure(figsize=(7, 4))
plt.hist(observations, bins=20, density=True, label="measurements")
plt.axvline(a, color="red", linestyle="--", linewidth=1.5, label=rf"$a = {a:.0f}$")
plt.axvline(b, color="red", linestyle="--", linewidth=1.5, label=rf"$b = {b:.0f}$")
plt.axvline(
    a + b,
    color="red",
    linestyle="--",
    linewidth=1.5,
    label=rf"expected $a+b = {a:.0f}+{b:.0f} = {a + b:.0f}$",
)
plt.axvline(
    observations.mean(),
    color="orange",
    linestyle="-",
    linewidth=1.5,
    label=f"mean = {observations.mean():.3f}",
)
plt.xlabel(r"measured $p_1$")
plt.ylabel("probability density")
plt.xlim(1.5, 5.5)
plt.title(rf"Addition via CV CZ gate: $2 + 3 = 5$ ($r = {r}$)")
plt.legend(loc="upper left")
plt.tight_layout()
plt.show()

The histogram shows a distribution centered at $p_1 \approx 5$. The theoretical spread of the distribution is $e^{-r}$; stronger squeezing improves accuracy.

### 2.4 Squeezing vs. Accuracy

We vary the squeezing parameter $r$ and observe how the variance of the measurement outcomes scales. Theoretical calculations show that the variance of the $p$ measurement for mode 1 after the CZ gate is given by $\sigma^2 = 2 \cdot (e^{-r}/\sqrt{2})^2 = e^{-2r}$; thus, the standard deviation is $e^{-r}$.

In [ ]:
print(f"{'r':>4}  {'mean':>8}  {'std':>10}  {'theory std':>14}")
print("-" * 44)
r_vals = np.linspace(0.5, 3.5, 12)
stds = []
for r_val in r_vals:
    obs = add_with_cv_cz(a, b, r=r_val, n_shots=500)
    std = obs.std()
    print(f"{r_val:>4.1f}  {obs.mean():>8.3f}  {std:>10.4f}  {np.exp(-r_val):>14.4f}")
    stds.append(std)

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(r_vals, stds, "o-", label="measured std")
plt.plot(r_vals, np.exp(-r_vals), "--", color="gray", label=r"theory $e^{-r}$")
plt.xlabel("squeezing parameter $r$")
plt.ylabel("standard deviation")
plt.title("Squeezing vs. addition accuracy")
plt.legend()
plt.tight_layout()
plt.show()

In the limit $r \to \infty$, the standard deviation approaches zero, converging to an ideal scalar addition.

---

## Summary

In this tutorial, we demonstrated the following.

1. **Encoding values into continuous variables**: In CV quantum computing, numerical values can be carried by the continuous quadratures $\hat{x}$ and $\hat{p}$ of bosonic systems. In this demo, values $a$ and $b$ are encoded as the expectation values $\langle \hat{p} \rangle = a$ and $\langle \hat{x} \rangle = b$ of quantum states.
2. **Addition via the CV CZ gate**: Applying CZ(1) to $\ket{p \approx a} \otimes \ket{x \approx b}$ shifts the mean of $p$ in mode 1 to $a + b$.
3. **Approximate implementation with squeezed states**: Since ideal $x$- and $p$-quadrature eigenstates are physically unrealizable, we approximate them with squeezed states. The stronger the squeezing, the smaller the measurement error (on the scale of $e^{-r}$), and the closer the result is to classical addition.

---

## References

- \[1\] E. H. Kennard, Zur Quantenmechanik einfacher Bewegungstypen, [*Z. Phys.* **44**, 326 (1927)](https://doi.org/10.1007/BF01391200).
- \[2\] H. P. Robertson, The Uncertainty Principle, [*Phys. Rev.* **34**, 163 (1929)](https://doi.org/10.1103/PhysRev.34.163).